# Relaxed staging on `bram(7,5)` at the tight target

Single-config experiment: the headline `mark_bram(7,5)` promotion
(pair-packed, **no DSP**) built twice from the same trace — once with the
default stage-isolation rules, once with `relaxed=True` (combinational ops
may share clocked stages; only readers of a clocked op's un-registered
address input are evicted) — at the tight `T = 2.5 ns` Vivado target.

Produces `relaxed_bram_7_5_metrics.csv` and `SUMMARY.md` with the
default-vs-relaxed comparison (depth, LUT, LUT-as-memory, FF, BRAM, WNS,
Fmax).

**Requires** the da4ml fork built at a commit that includes
`mark_bram(relaxed=...)` / `to_pipeline(relaxed=...)` (after `df88697`).

**Note on `LATENCY_CUTOFF`:** set to 3.0 here. The thesis tight-target
rerun (`tab:tight-target`) used 2.0 at the same 2.5 ns clock; set
`LATENCY_CUTOFF = 2.0` below for a directly comparable point. With 3.0
the per-stage combinational budget exceeds the clock period, so expect
negative WNS — the comparison between the two stagings is still valid
(same cutoff on both sides).

**To run:** edit the config cell (Vivado settings path, dataset dir), then
run all. Two Vivado jobs, about 1–2 h total.

In [1]:
import os
os.environ.setdefault('KERAS_BACKEND', 'jax')
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.5')

import sys
from pathlib import Path

CHECKPOINT = Path('../../../models/jsc_plf/64-16/'
                  'epoch=4433-val_acc=0.817-ebops=40646-val_loss=0.534.keras')
PARTICLES, FEATURES = 64, 16
PART_NAME   = 'xczu7ev-ffvc1156-2-e'
MODEL_NAME  = 'jsc'

CLOCK_PERIOD      = 2.5     # ns -> 400 MHz target (tight)
CLOCK_UNCERTAINTY = 0.0
LATENCY_CUTOFF    = 2     # thesis tight-target used 2.0; see header note
BW_IN, BW_OUT     = 7, 5    # the headline predicate; no DSP passes anywhere
PAIR_WITHIN_STAGE = True
TRACE_BATCH       = 2048
N_BITEXACT        = 256
RUN_VIVADO        = True
MAX_VIVADO_PAR    = 2
VIVADO_SETTINGS   = '/tools/Xilinx/2025.1/Vivado/settings64.sh'

HELPERS  = Path('../../HGQ-LUT_jsc-plf_64p-16f').resolve()  # model.py, data.py
DATA_DIR = Path('../../../dataset/jsc_plf').resolve()                    # from prepare_datasets.sh
NB_DIR   = Path('.').resolve()
if str(HELPERS) not in sys.path:
    sys.path.insert(0, str(HELPERS))

VARIANTS = ['default', 'relaxed']

print(f'Config       : bram({BW_IN},{BW_OUT}) pair={PAIR_WITHIN_STAGE}, no DSP')
print(f'Part / clock : {PART_NAME}  T={CLOCK_PERIOD} ns  cutoff={LATENCY_CUTOFF} ns')
print(f'Variants     : {VARIANTS}')
print(f'Output dir   : {NB_DIR}')

Config       : bram(7,5) pair=True, no DSP
Part / clock : xczu7ev-ffvc1156-2-e  T=2.5 ns  cutoff=2 ns
Variants     : ['default', 'relaxed']
Output dir   : /home/kevin/Dev/Imperial/HGQ-LUT-AE/test_notebooks


Load the cached pre-traced checkpoint (traces min/max on first run only).

In [2]:
import keras
import numpy as np
from model import SameDim0
from data import get_data
from hgq.utils import trace_minmax

raw_ckpt = CHECKPOINT.expanduser().resolve()
traced   = raw_ckpt.parent / f'model_traced_n{PARTICLES}_f{FEATURES}.keras'
if not traced.exists():
    print('Running trace_minmax (first run only)...')
    m0 = keras.models.load_model(raw_ckpt, compile=False, custom_objects={'SameDim0': SameDim0})
    (Xtr, _), (Xv, _), _ = get_data(DATA_DIR, PARTICLES, FEATURES == 3)
    trace_minmax(m0, Xtr, batch_size=TRACE_BATCH, reset=True,  verbose=True)
    trace_minmax(m0, Xv,  batch_size=TRACE_BATCH, reset=False, verbose=True)
    m0.save(traced)
model = keras.models.load_model(traced, compile=False, custom_objects={'SameDim0': SameDim0})
model_stem = raw_ckpt.stem
print(f'Loaded N={model.input_shape[1]} F={model.input_shape[-1]}')

(_, _), (Xval, _), _ = get_data(DATA_DIR, PARTICLES, FEATURES == 3)
X_BE = np.asarray(Xval)[:N_BITEXACT]
print(f'Bit-exact test batch: {X_BE.shape}')

Loaded N=64 F=16
Bit-exact test batch: (256, 64, 16)


Trace once; the unmarked `comb_base.predict` is the bit-exact reference.

In [3]:
from da4ml.converter import trace_model
from da4ml.trace import HWConfig, comb_trace, mark_bram
from da4ml.trace.pipeline import to_pipeline

HW_CONFIG      = HWConfig(1, -1, -1)
SOLVER_OPTIONS = {'hard_dc': 2}

def _trace_base():
    inp, out = trace_model(model, hwconf=HW_CONFIG, solver_options=SOLVER_OPTIONS, verbose=False)
    return comb_trace(inp, out)

comb_base = _trace_base()
max_lat = max(op.latency for op in comb_base.ops)
print(f'max(comb_base.latency) = {max_lat}')

BASELINE_PRED = comb_base.predict(X_BE, n_threads=-1)
print(f'baseline pred shape {BASELINE_PRED.shape} — reference for bit-exact gate')

PREDICATE = lambda bw_in, bw_out: bw_in >= BW_IN and bw_out >= BW_OUT

max(comb_base.latency) = 33.0
baseline pred shape (256, 5) — reference for bit-exact gate


Build both variants from the same trace: mark, pipeline, SW bit-exact gate, write RTL. Print the stage maps — under `relaxed` the clocked stages should contain combinational ops instead of evicting them.

In [4]:
import shutil, re
from da4ml.codegen import RTLModel

def _stage_stats(pipe):
    rows = []
    for sol in pipe.solutions:
        n_bram = sum(1 for op in sol.ops if op.opcode == 8 and ((op.data >> 32) & 1))
        n_comb = sum(1 for op in sol.ops if op.opcode != -1 and not (op.opcode == 8 and ((op.data >> 32) & 1)))
        rows.append((n_bram, n_comb))
    return rows

records = {}
for variant in VARIANTS:
    relaxed = variant == 'relaxed'
    out_path = NB_DIR / f'bram_{BW_IN}_{BW_OUT}__{variant}' / model_stem
    if out_path.parent.exists():
        shutil.rmtree(out_path.parent)
    out_path.mkdir(parents=True, exist_ok=True)

    comb = _trace_base()
    comb = mark_bram(comb, predicate=PREDICATE, latency_cutoff=LATENCY_CUTOFF,
                     pair_within_stage=PAIR_WITHIN_STAGE, relaxed=relaxed)
    pipe = to_pipeline(comb, LATENCY_CUTOFF, verbose=False, relaxed=relaxed)

    pred = comb.predict(X_BE, n_threads=-1)
    be_ok = bool(np.all(pred == BASELINE_PRED))
    assert be_ok, f'{variant}: BIT-EXACT FAILURE vs unmarked baseline'

    rtl = RTLModel(pipe, MODEL_NAME, str(out_path), clock_period=CLOCK_PERIOD,
                   part_name=PART_NAME, clock_uncertainty=CLOCK_UNCERTAINTY)
    rtl.write()

    stats = _stage_stats(pipe)
    n_bram_ops = sum(nb for nb, _ in stats)
    n_shared   = sum(1 for nb, nc in stats if nb > 0 and nc > 0)
    records[variant] = dict(prj_dir=out_path, stages=len(pipe.solutions),
                            n_bram_ops=n_bram_ops, n_shared_stages=n_shared,
                            comb=comb, pipe=pipe)
    print(f'--- {variant}: {len(pipe.solutions)} stages, {n_bram_ops} BRAM lookups, '
          f'{n_shared} stage(s) shared by BRAM+comb, bit-exact OK')
    for i, (nb, nc) in enumerate(stats):
        tag = '  <-- shared' if nb > 0 and nc > 0 else (' <-- clocked' if nb > 0 else '')
        print(f'    stage{i:2d}: bram={nb:4d} comb={nc:5d}{tag}')

d_def, d_rlx = records['default']['stages'], records['relaxed']['stages']
print(f'\nDepth: default={d_def}  relaxed={d_rlx}  (saved {d_def - d_rlx} cycles)')

--- default: 30 stages, 344 BRAM lookups, 1 stage(s) shared by BRAM+comb, bit-exact OK
    stage 0: bram=   0 comb= 8892
    stage 1: bram= 207 comb=    0 <-- clocked
    stage 2: bram=   0 comb= 2728
    stage 3: bram=   1 comb=   10  <-- shared
    stage 4: bram=   0 comb= 3865
    stage 5: bram=  24 comb=    0 <-- clocked
    stage 6: bram=   0 comb= 1318
    stage 7: bram=  18 comb=    0 <-- clocked
    stage 8: bram=   0 comb=  432
    stage 9: bram=   0 comb=  129
    stage10: bram=   9 comb=    0 <-- clocked
    stage11: bram=   0 comb=   71
    stage12: bram=   5 comb=    0 <-- clocked
    stage13: bram=   0 comb=   25
    stage14: bram=   0 comb=  689
    stage15: bram=   0 comb=  967
    stage16: bram=   0 comb=  131
    stage17: bram=   0 comb=  174
    stage18: bram=  24 comb=    0 <-- clocked
    stage19: bram=   0 comb=  233
    stage20: bram=  18 comb=    0 <-- clocked
    stage21: bram=   0 comb=   70
    stage22: bram=   0 comb=  116
    stage23: bram=   6 comb=    0 <

Verilator gate on both variants: generated RTL must match `comb.predict` bit-for-bit (this is the cycle-alignment proof; the SW gate above only checks the dataflow).

In [5]:
import tempfile

for variant in VARIANTS:
    r = records[variant]
    scratch = Path(tempfile.mkdtemp(prefix=f'relaxed75_veri_{variant}_'))
    try:
        rtl = RTLModel(r['pipe'], MODEL_NAME, str(scratch), clock_period=CLOCK_PERIOD,
                       part_name=PART_NAME, clock_uncertainty=CLOCK_UNCERTAINTY)
        rtl.write()
        for _ in range(8):
            try:
                rtl._compile(openmp=False, nproc=4); break
            except RuntimeError:
                pass
        hw = rtl.predict(X_BE)
        sw = r['comb'].predict(X_BE, n_threads=-1)
        ndiff = int(np.sum(hw != sw))
        print(f'  {variant:<8s}: rtl(verilator) vs comb(SW) diffs = {ndiff} / {hw.size}')
        assert ndiff == 0, f'{variant}: Verilator != SW IR!'
    finally:
        shutil.rmtree(scratch, ignore_errors=True)
print('\n\u2713 Verilator bit-accuracy passed on both variants.')

  default : rtl(verilator) vs comb(SW) diffs = 0 / 1280
  relaxed : rtl(verilator) vs comb(SW) diffs = 0 / 1280

✓ Verilator bit-accuracy passed on both variants.


Run Vivado on both projects (~1–2 h).

In [6]:
import shlex, subprocess, time
from collections import deque

def start_job(tcl):
    rd = tcl.parent
    of = open(rd / 'vivado_stdout.log', 'w'); ef = open(rd / 'vivado_stderr.log', 'w')
    cmd = (f'set -e\nsource {shlex.quote(VIVADO_SETTINGS)}\n'
           f'vivado -mode batch -source {shlex.quote(str(tcl.resolve()))} -nojournal -nolog\n')
    p = subprocess.Popen(['bash', '-lc', cmd], cwd=str(rd), stdout=of, stderr=ef, text=True)
    return dict(rd=rd, p=p, of=of, ef=ef, t0=time.time())

if RUN_VIVADO:
    tcls = [records[v]['prj_dir'] / 'build_vivado_prj.tcl' for v in VARIANTS]
    q = deque(tcls); running = []; done = []
    def drain():
        while q and len(running) < MAX_VIVADO_PAR:
            j = start_job(q.popleft()); running.append(j)
            print(f'[START] {j["rd"].parent.name}')
    drain()
    while running:
        time.sleep(1.0)
        for j in list(running):
            rc = j['p'].poll()
            if rc is None: continue
            j['of'].close(); j['ef'].close()
            print(f'[{"OK" if rc==0 else f"rc={rc}"}] {j["rd"].parent.name}  t={time.time()-j["t0"]:.0f}s')
            done.append((j['rd'].parent.name, rc)); running.remove(j); drain()
    print(f'\nVivado done {len(done)}/{len(tcls)}; nonzero rc: {[d for d in done if d[1]!=0] or "none"}')
else:
    print('RUN_VIVADO=False — skipping.')

[START] bram_7_5__default
[START] bram_7_5__relaxed
[OK] bram_7_5__relaxed  t=397s
[OK] bram_7_5__default  t=428s

Vivado done 2/2; nonzero rc: none


Parse post-place / post-route reports into the comparison table.

In [7]:
import json
import pandas as pd

def _used(t, label):
    m = re.search(r'^\|\s*' + re.escape(label) + r'\s*\|\s*([\d.]+)\s*\|', t, re.MULTILINE)
    return float(m.group(1)) if m else float('nan')

def _wns(path):
    if not path.exists(): return float('nan')
    m = re.search(r'WNS\(ns\)[^\n]*\n[^\n]*\n\s*([-\d.]+)', path.read_text())
    return float(m.group(1)) if m else float('nan')

rows = []
for variant in VARIANTS:
    r = records[variant]
    rep = r['prj_dir'] / 'output_jsc' / 'reports'
    util = rep / 'jsc_post_route_util.rpt'
    t = util.read_text() if util.exists() else ''
    wpp = _wns(rep / 'jsc_post_place_timing.rpt')
    wpr = _wns(rep / 'jsc_post_route_timing.rpt')
    fpp = 1000.0/(CLOCK_PERIOD-wpp) if wpp==wpp and (CLOCK_PERIOD-wpp)>0 else float('nan')
    fpr = 1000.0/(CLOCK_PERIOD-wpr) if wpr==wpr and (CLOCK_PERIOD-wpr)>0 else float('nan')
    status = ('no_reports' if not util.exists() else
              'met' if wpr==wpr and wpr >= 0 else
              'timing_fail' if wpr==wpr else 'no_timing')
    rows.append(dict(variant=variant, stages=r['stages'],
                     shared_stages=r['n_shared_stages'], n_bram_lookups=r['n_bram_ops'],
                     lut_total=_used(t,'CLB LUTs'), lut_logic=_used(t,'LUT as Logic'),
                     lut_memory=_used(t,'LUT as Memory'), ff=_used(t,'CLB Registers'),
                     bram_tile=_used(t,'Block RAM Tile'), dsp=_used(t,'DSPs'),
                     wns_pp=wpp, fmax_pp_MHz=fpp, wns_pr=wpr, fmax_pr_MHz=fpr,
                     latency_ns=r['stages']*(CLOCK_PERIOD-wpr) if wpr==wpr else float('nan'),
                     status=status))

df = pd.DataFrame(rows)
df.to_csv(NB_DIR / 'relaxed_bram_7_5_metrics.csv', index=False)
print(df.to_string(index=False))
assert (df['dsp'].fillna(0) == 0).all(), 'DSP must be zero in this experiment'

variant  stages  shared_stages  n_bram_lookups  lut_total  lut_logic  lut_memory      ff  bram_tile  dsp  wns_pp  fmax_pp_MHz  wns_pr  fmax_pr_MHz  latency_ns      status
default      30              1             344    33094.0    30101.0      2993.0 45236.0       99.5  0.0  -0.091   385.951370  -0.155   376.647834      79.650 timing_fail
relaxed      22             11             344    32162.0    30062.0      2100.0 27836.0       99.5  0.0  -0.101   384.467512  -0.258   362.581581      60.676 timing_fail


Write `SUMMARY.md`.

In [8]:
d, x = df[df.variant=='default'].iloc[0], df[df.variant=='relaxed'].iloc[0]
summary = f"""# test_notebooks/relaxed_bram_7_5_tight — relaxed staging on the headline predicate

Single-config default-vs-relaxed comparison of `mark_bram({BW_IN},{BW_OUT})`
(pair-packed, zero DSP) at `T = {CLOCK_PERIOD}` ns, `latency_cutoff = {LATENCY_CUTOFF}` ns,
on `{PART_NAME}`. Relaxed staging lets combinational ops share clocked
stages (`mark_bram(relaxed=True)` + `to_pipeline(relaxed=True)`); only
readers of a clocked op's un-registered address input are still evicted.

Note: the thesis tight-target rerun used `latency_cutoff = 2.0` at the same
clock; rerun with that value for a row directly comparable to `tab:tight-target`.

| metric | default | relaxed | delta |
|---|---|---|---|
| pipeline depth (cycles) | {d.stages:.0f} | {x.stages:.0f} | {x.stages-d.stages:+.0f} |
| stages shared (BRAM+comb) | {d.shared_stages:.0f} | {x.shared_stages:.0f} | — |
| LUT total | {d.lut_total:.0f} | {x.lut_total:.0f} | {x.lut_total-d.lut_total:+.0f} |
| LUT as memory | {d.lut_memory:.0f} | {x.lut_memory:.0f} | {x.lut_memory-d.lut_memory:+.0f} |
| FF | {d.ff:.0f} | {x.ff:.0f} | {x.ff-d.ff:+.0f} |
| BRAM tile | {d.bram_tile:.1f} | {x.bram_tile:.1f} | {x.bram_tile-d.bram_tile:+.1f} |
| DSP | {d.dsp:.0f} | {x.dsp:.0f} | — |
| WNS post-place (ns) | {d.wns_pp:+.3f} | {x.wns_pp:+.3f} | — |
| WNS post-route (ns) | {d.wns_pr:+.3f} | {x.wns_pr:+.3f} | — |
| Fmax post-route (MHz) | {d.fmax_pr_MHz:.1f} | {x.fmax_pr_MHz:.1f} | {x.fmax_pr_MHz-d.fmax_pr_MHz:+.1f} |
| end-to-end latency (ns) | {d.latency_ns:.1f} | {x.latency_ns:.1f} | {x.latency_ns-d.latency_ns:+.1f} |
| status | {d.status} | {x.status} | |

Both variants are bit-exact to the unmarked baseline at the SW-IR level
(256 validation samples) and bit-exact RTL-vs-IR under Verilator.
"""
(NB_DIR / 'SUMMARY.md').write_text(summary)
print(summary)

# test_notebooks/relaxed_bram_7_5_tight — relaxed staging on the headline predicate

Single-config default-vs-relaxed comparison of `mark_bram(7,5)`
(pair-packed, zero DSP) at `T = 2.5` ns, `latency_cutoff = 2` ns,
on `xczu7ev-ffvc1156-2-e`. Relaxed staging lets combinational ops share clocked
stages (`mark_bram(relaxed=True)` + `to_pipeline(relaxed=True)`); only
readers of a clocked op's un-registered address input are still evicted.

Note: the thesis tight-target rerun used `latency_cutoff = 2.0` at the same
clock; rerun with that value for a row directly comparable to `tab:tight-target`.

| metric | default | relaxed | delta |
|---|---|---|---|
| pipeline depth (cycles) | 30 | 22 | -8 |
| stages shared (BRAM+comb) | 1 | 11 | — |
| LUT total | 33094 | 32162 | -932 |
| LUT as memory | 2993 | 2100 | -893 |
| FF | 45236 | 27836 | -17400 |
| BRAM tile | 99.5 | 99.5 | +0.0 |
| DSP | 0 | 0 | — |
| WNS post-place (ns) | -0.091 | -0.101 | — |
| WNS post-route (ns) | -0.155 | -0.258 | — |
| F

In [9]:
# ── Optional: cocotb gate — bit-exactness AND latency-exactness, both variants ──
# Drives jsc_wrapper through Verilator under cocotb using the shared driver
# (cocotb_jsc_test_runner.py in HELPERS, already on sys.path). Two tests per
# variant: `correctness_and_latency` (sample-aligned readback at cycles
# [latency .. latency+N-1], bit-exact vs comb.predict) and `latency_boundary`
# (probe sample, first matching output cycle must equal the claimed latency).
# Slot geometry is derived from the IR (same formula as codegen's
# hetero_io_map): slot_w = max(i+k)+max(f), binary point at slot bit max(f).
import xml.etree.ElementTree as ET
from cocotb_tools.runner import get_runner

N_COCOTB = 16
RNG_COCOTB = np.random.default_rng(0)

def _slot_geom(kifs):
    k, i, f = kifs[0].astype(int), kifs[1].astype(int), kifs[2].astype(int)
    lsb_ref = int(f.max())
    slot_w = int((i + k).max()) + lsb_ref
    return slot_w, lsb_ref

def _pack(int_vals, kifs, slot_w, lsb_ref):
    out = 0
    for j, v in enumerate(int_vals):
        k, i, f = int(kifs[0, j]), int(kifs[1, j]), int(kifs[2, j])
        bw = k + i + f
        if bw <= 0:
            continue
        out |= ((int(v) & ((1 << bw) - 1)) << (lsb_ref - f)) << (slot_w * j)
    return out

for variant in VARIANTS:
    r = records[variant]
    comb = r['comb']
    src_dir = r['prj_dir'] / 'src'
    sim_dir = NB_DIR / f'_cocotb_sim_{variant}'
    sim_build = sim_dir / 'sim_build'
    sim_build.mkdir(parents=True, exist_ok=True)

    inp_kifs = np.asarray(comb.inp_kifs)
    out_kifs = np.asarray(comb.out_kifs)
    in_slot_w, in_lsb = _slot_geom(inp_kifs)
    out_slot_w, out_lsb = _slot_geom(out_kifs)
    n_in, n_out = inp_kifs.shape[1], out_kifs.shape[1]

    # Random integer vectors in each input's valid two's-complement range.
    int_inputs = np.zeros((N_COCOTB, n_in), dtype=np.int64)
    for j in range(n_in):
        k, i, f = (int(inp_kifs[0, j]), int(inp_kifs[1, j]), int(inp_kifs[2, j]))
        bw = k + i + f
        if bw <= 0:
            continue
        lo, hi = (-(1 << (bw - 1)), (1 << (bw - 1)) - 1) if k == 1 else (0, (1 << bw) - 1)
        int_inputs[:, j] = RNG_COCOTB.integers(lo, hi + 1, size=N_COCOTB, dtype=np.int64)

    floats = int_inputs.astype(np.float64) * (2.0 ** -inp_kifs[2].astype(np.float64))[None, :]
    ref = comb.predict(floats.reshape(N_COCOTB, PARTICLES, FEATURES), n_threads=1)
    int_expected = np.zeros((N_COCOTB, n_out), dtype=np.int64)
    for j in range(n_out):
        int_expected[:, j] = np.round(ref[:, j] * (2.0 ** int(out_kifs[2, j]))).astype(np.int64)

    np.savez(sim_dir / 'test_vectors.npz',
             model_inp=np.array([_pack(int_inputs[s], inp_kifs, in_slot_w, in_lsb)
                                 for s in range(N_COCOTB)], dtype=object),
             expected=int_expected)
    (sim_dir / 'test_config.json').write_text(json.dumps({
        'latency': int(r['stages']), 'clock_period_ns': CLOCK_PERIOD,
        'out_bws': (out_kifs.sum(axis=0)).astype(int).tolist(),
        'out_fs': out_kifs[2].astype(int).tolist(),
        'out_ks': out_kifs[0].astype(int).tolist(),
        'out_slot_w': out_slot_w, 'out_lsb_ref': out_lsb}, indent=2))
    for mf in (src_dir / 'memfiles').glob('*.mem'):
        shutil.copy(mf, sim_dir / mf.name)

    sources = sorted(src_dir.glob('*.v')) + sorted((src_dir / 'static').glob('*.v'))
    runner = get_runner('verilator')
    runner.build(sources=[str(s) for s in sources], hdl_toplevel='jsc_wrapper',
                 build_args=['-Wno-fatal', '-Wno-WIDTHEXPAND', '-Wno-WIDTHTRUNC',
                             '-Wno-UNUSEDSIGNAL', '-Wno-CASEINCOMPLETE'],
                 always=True, build_dir=str(sim_build))
    results_xml = runner.test(hdl_toplevel='jsc_wrapper',
                              test_module='cocotb_jsc_test_runner',
                              build_dir=str(sim_build), test_dir=str(sim_dir),
                              extra_env={'TEST_DATA_DIR': str(sim_dir)})

    failed = [tc.get('name') for ts in ET.parse(results_xml).getroot().iter('testsuite')
              for tc in ts.iter('testcase')
              if tc.find('failure') is not None or tc.find('error') is not None]
    assert not failed, f'{variant}: cocotb FAILED: {failed}'
    print(f'{variant:<8s}: cocotb correctness_and_latency + latency_boundary PASSED '
          f'(latency = {r["stages"]} cycles, {N_COCOTB} samples)')

print('\n\u2713 cocotb: bit- and latency-exactness confirmed on both variants.')


- V e r i l a t i o n   R e p o r t: Verilator 5.032 2025-01-01 rev (Debian 5.032-1)
- Verilator: Built from 104.515 MB sources in 44 modules, into 152.891 MB in 79 C++ files needing 0.003 MB
- Verilator: Walltime 16.458 s (elab=1.456, cvt=13.839, bld=0.000); cpu 16.444 s on 1 threads; alloced 715.660 MB
make: Entering directory '/home/kevin/Dev/Imperial/HGQ-LUT-AE/test_notebooks/_cocotb_sim_default/sim_build'
g++  -I.  -MMD -I/usr/share/verilator/include -I/usr/share/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TIMING=0 -DVM_TRACE=0 -DVM_TRACE_FST=0 -DVM_TRACE_VCD=0 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-shadow -Wno-sign-compare -Wno-tautological-compare -Wno-uninitialized -Wno-unused-but-set-parameter -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable      -Os  -c -o verilator.o /home/kevin/venvs/ml5090-312/lib/python3.12/site-packages/cocotb/share/lib/verilator/verilator.cpp
g++ -Os  -I.  -MMD -I/usr/share/verilator/include -I/usr